In [ ]:
import os
import math
import time
import random
import zipfile
from typing import Dict, Any, Tuple, List

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


# =============================================================================
# 0. Matplotlib style
# =============================================================================
plt.rcParams.update({
    "pdf.fonttype": 42,
    "font.family": "serif",
    "font.serif": ["Liberation Serif", "FreeSerif", "serif"],
    "font.size": 10,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "mathtext.fontset": "stix",
})


# =============================================================================
# 1. Unified Configuration
# =============================================================================
CFG: Dict[str, Any] = {
    # reproducibility / device / dtype
    "seed": 0,
    "seeds": [0, 1, 2, 3, 4],
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "dtype": torch.float32,

    # dynamics / cost
    "a0": -0.5,
    "b0": 1.0,
    "sigma": 0.2,
    "R": 0.1,
    "S0": -5.0,

    # time grid
    "T": 5.0,
    "N_steps": 50,

    # distributed delay kernel
    "delta": 3.0,
    "H_delay": 20,
    "a1_scale": -10.0,
    "a1_lambda": 1.0,

    # initial history
    "x0": 10.0,
    "x_hist_const": 10.0,

    # Method 1: LSTM-DPO warm-up
    "warmup_batch_size": 256,
    "warmup_iters": 10000,
    "lr_pg": 3e-4,

    # Stage 2 projection
    "N_mc_stage2": 4096,

    # network
    "hidden": 64,

    # output
    "outdir": "benchmark3_multiseed_figures",
}

DEVICE = torch.device(CFG["device"])
torch.set_default_dtype(CFG["dtype"])
OUTDIR = CFG["outdir"]
os.makedirs(OUTDIR, exist_ok=True)


# =============================================================================
# 2. Utilities
# =============================================================================
def cfg_for_seed(cfg: Dict[str, Any], seed: int) -> Dict[str, Any]:
    out = dict(cfg)
    out["seed"] = int(seed)
    return out


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_dt(cfg: Dict[str, Any]) -> float:
    return float(cfg["T"]) / float(cfg["N_steps"])


# =============================================================================
# 3. Physics helpers
# =============================================================================
def build_a1_kernel_np(cfg: Dict[str, Any]) -> np.ndarray:
    H = int(cfg["H_delay"])
    delta = float(cfg["delta"])
    dtheta = delta / H

    theta = -delta + dtheta * (np.arange(H) + 0.5)
    a1_vals = float(cfg["a1_scale"]) * np.exp(float(cfg["a1_lambda"]) * theta)

    w = a1_vals * dtheta
    w = w[::-1].copy()

    return w.astype(np.float32)


def build_a1_kernel_torch(
    cfg: Dict[str, Any],
    device: torch.device,
    dtype: torch.dtype,
) -> torch.Tensor:
    return torch.tensor(build_a1_kernel_np(cfg), device=device, dtype=dtype)


def init_hist_torch(
    cfg: Dict[str, Any],
    batch_size: int,
    device: torch.device,
    dtype: torch.dtype,
) -> torch.Tensor:
    H = int(cfg["H_delay"])
    x_hist = torch.full(
        (batch_size, H),
        float(cfg["x_hist_const"]),
        device=device,
        dtype=dtype,
    )
    x_hist[:, 0] = float(cfg["x0"])
    return x_hist


def delay_integral_torch(w: torch.Tensor, x_hist: torch.Tensor) -> torch.Tensor:
    return torch.sum(w * x_hist, dim=1)


def shift_hist_torch(x_hist: torch.Tensor, x_new: torch.Tensor) -> torch.Tensor:
    return torch.cat([x_new.unsqueeze(1), x_hist[:, :-1]], dim=1)


def env_step_torch(
    cfg: Dict[str, Any],
    w: torch.Tensor,
    x_hist: torch.Tensor,
    v: torch.Tensor,
    dt: float,
    stochastic: bool = True,
) -> Tuple[torch.Tensor, torch.Tensor]:
    x_curr = x_hist[:, 0]
    integ = delay_integral_torch(w, x_hist)
    drift = float(cfg["a0"]) * x_curr + integ + float(cfg["b0"]) * v

    if stochastic:
        dB = math.sqrt(dt) * torch.randn_like(x_curr)
        x_next = x_curr + drift * dt + float(cfg["sigma"]) * dB
    else:
        x_next = x_curr + drift * dt

    x_hist_next = shift_hist_torch(x_hist, x_next)
    return x_hist_next, x_next


def running_cost_torch(cfg: Dict[str, Any], v: torch.Tensor, dt: float) -> torch.Tensor:
    return float(cfg["R"]) * (v ** 2) * dt


def terminal_cost_torch(cfg: Dict[str, Any], x_T: torch.Tensor) -> torch.Tensor:
    return float(cfg["S0"]) * x_T


# =============================================================================
# 4. Method 1: LSTM-DPO warm-up
# =============================================================================
class LSTMPolicy(nn.Module):
    def __init__(self, input_size: int = 2, hidden_size: int = 64):
        super().__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTMCell(input_size, hidden_size)
        self.head = nn.Linear(hidden_size, 1)
        nn.init.constant_(self.head.bias, 1.0)

    def init_hidden(
        self,
        batch_size: int,
        device: torch.device,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        h0 = torch.zeros(batch_size, self.hidden_size, device=device)
        c0 = torch.zeros(batch_size, self.hidden_size, device=device)
        return h0, c0

    def forward_step(
        self,
        t_norm: torch.Tensor,
        x_t: torch.Tensor,
        h: torch.Tensor,
        c: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        inp = torch.stack([t_norm, x_t], dim=-1)
        h_next, c_next = self.lstm(inp, (h, c))
        v_raw = self.head(h_next).squeeze(-1)
        v = torch.relu(v_raw)
        return v, h_next, c_next


def simulate_pg_batch(
    policy: LSTMPolicy,
    cfg: Dict[str, Any],
    batch_size: int,
    detach_policy: bool = False,
    stochastic: bool = True,
) -> torch.Tensor:
    dt = get_dt(cfg)
    N = int(cfg["N_steps"])
    w = build_a1_kernel_torch(cfg, device=DEVICE, dtype=torch.float32)

    x_hist = init_hist_torch(cfg, batch_size, device=DEVICE, dtype=torch.float32)
    h, c = policy.init_hidden(batch_size, DEVICE)

    cost = torch.zeros(batch_size, device=DEVICE, dtype=torch.float32)

    for n in range(N):
        t_norm = torch.full(
            (batch_size,),
            (n * dt) / float(cfg["T"]),
            device=DEVICE,
            dtype=torch.float32,
        )
        x_curr = x_hist[:, 0]

        if detach_policy:
            with torch.no_grad():
                v, h, c = policy.forward_step(t_norm, x_curr, h, c)
        else:
            v, h, c = policy.forward_step(t_norm, x_curr, h, c)

        cost = cost + running_cost_torch(cfg, v, dt)
        x_hist, _ = env_step_torch(cfg, w, x_hist, v, dt, stochastic=stochastic)

    x_T = x_hist[:, 0]
    cost = cost + terminal_cost_torch(cfg, x_T)

    return cost.mean()


def warmup_train_pg(policy: LSTMPolicy, cfg: Dict[str, Any]) -> LSTMPolicy:
    print("\n[Method 1] Starting PG Warm-up for LSTM-DPO...")
    policy.to(DEVICE).train()

    opt = optim.Adam(policy.parameters(), lr=float(cfg["lr_pg"]))

    for it in range(int(cfg["warmup_iters"])):
        opt.zero_grad(set_to_none=True)

        J = simulate_pg_batch(
            policy,
            cfg,
            batch_size=int(cfg["warmup_batch_size"]),
            detach_policy=False,
            stochastic=True,
        )

        J.backward()
        opt.step()

        if (it + 1) % 500 == 0:
            print(f"  Iter {it+1}, Cost J = {J.item():.4f}")

    return policy


@torch.no_grad()
def rollout_pg_deterministic(
    policy: LSTMPolicy,
    cfg: Dict[str, Any],
) -> Dict[str, List[torch.Tensor]]:
    dt = get_dt(cfg)
    N = int(cfg["N_steps"])
    w = build_a1_kernel_torch(cfg, device=DEVICE, dtype=torch.float32)

    batch_size = 1
    x_hist = init_hist_torch(cfg, batch_size, device=DEVICE, dtype=torch.float32)
    h, c = policy.init_hidden(batch_size, DEVICE)

    xs, vs, hs, cs, x_hists = [], [], [], [], []

    for n in range(N):
        t_norm = torch.full(
            (batch_size,),
            (n * dt) / float(cfg["T"]),
            device=DEVICE,
            dtype=torch.float32,
        )
        x_curr = x_hist[:, 0]

        xs.append(x_curr.clone())
        hs.append(h.clone())
        cs.append(c.clone())
        x_hists.append(x_hist.clone())

        v, h, c = policy.forward_step(t_norm, x_curr, h, c)
        vs.append(v.clone())

        x_hist, _ = env_step_torch(cfg, w, x_hist, v, dt, stochastic=False)

    return {
        "xs": xs,
        "vs": vs,
        "hs": hs,
        "cs": cs,
        "x_hists": x_hists,
    }


# =============================================================================
# 5. PGDPO projection
# =============================================================================
def estimate_costate_from_step(
    policy: LSTMPolicy,
    cfg: Dict[str, Any],
    n0: int,
    x_hist0: torch.Tensor,
    h0: torch.Tensor,
    c0: torch.Tensor,
) -> torch.Tensor:
    dt = get_dt(cfg)
    N = int(cfg["N_steps"])
    M = int(cfg["N_mc_stage2"])

    w = build_a1_kernel_torch(cfg, device=DEVICE, dtype=torch.float32)

    x0_var = x_hist0[0, 0].detach().clone().requires_grad_(True)
    x_curr_batch = x0_var.expand(M)

    x_hist_rest = x_hist0[:, 1:].repeat(M, 1)
    x_hist = torch.cat([x_curr_batch.unsqueeze(1), x_hist_rest], dim=1)

    h = h0.repeat(M, 1)
    c = c0.repeat(M, 1)

    cost = torch.zeros(M, device=DEVICE, dtype=torch.float32)

    for n in range(n0, N):
        t_norm = torch.full(
            (M,),
            (n * dt) / float(cfg["T"]),
            device=DEVICE,
            dtype=torch.float32,
        )

        v, h, c = policy.forward_step(t_norm, x_hist[:, 0], h, c)

        cost = cost + running_cost_torch(cfg, v, dt)
        x_hist, _ = env_step_torch(cfg, w, x_hist, v, dt, stochastic=True)

    x_T = x_hist[:, 0]
    cost = cost + terminal_cost_torch(cfg, x_T)

    J_mean = cost.mean()

    grad_x0, = torch.autograd.grad(
        J_mean,
        x0_var,
        retain_graph=False,
        create_graph=False,
    )

    return grad_x0.detach()


def compute_pgdpo_projection_with_timing(
    policy_pg: LSTMPolicy,
    cfg: Dict[str, Any],
    rollout: Dict[str, List[torch.Tensor]],
) -> Dict[str, Any]:
    print("  Calculating PGDPO projections...")

    N = int(cfg["N_steps"])

    assert len(rollout["vs"]) == N
    assert len(rollout["x_hists"]) == N
    assert len(rollout["hs"]) == N
    assert len(rollout["cs"]) == N

    v_lstm: List[float] = []
    v_pgdpo: List[float] = []
    step_times: List[float] = []

    total_t0 = time.time()

    for n in range(N):
        step_t0 = time.time()

        v_net = float(rollout["vs"][n].item())

        lam = estimate_costate_from_step(
            policy_pg,
            cfg,
            n,
            rollout["x_hists"][n],
            rollout["hs"][n],
            rollout["cs"][n],
        )

        v_proj = max(
            0.0,
            -(float(cfg["b0"]) / (2.0 * float(cfg["R"]))) * float(lam.item()),
        )

        step_times.append(time.time() - step_t0)

        v_lstm.append(v_net)
        v_pgdpo.append(v_proj)

    total_time = time.time() - total_t0

    # IMPORTANT:
    # Do not append the last value to match the benchmark N+1 grid.
    # Controls are evaluated only on decision times t_0, ..., t_{N-1}.
    step_times_np = np.array(step_times, dtype=np.float64)

    timing = {
        "pgdpo_projection_total_sec": float(total_time),
        "pgdpo_projection_avg_step_sec": float(np.mean(step_times_np)) if len(step_times_np) > 0 else np.nan,
        "pgdpo_projection_min_step_sec": float(np.min(step_times_np)) if len(step_times_np) > 0 else np.nan,
        "pgdpo_projection_max_step_sec": float(np.max(step_times_np)) if len(step_times_np) > 0 else np.nan,
        "pgdpo_projection_step_times": step_times_np,
        "pgdpo_projection_n_steps": int(len(step_times_np)),
    }

    return {
        "v_lstm": np.array(v_lstm, dtype=np.float64),
        "v_pgdpo": np.array(v_pgdpo, dtype=np.float64),
        "timing": timing,
    }


# =============================================================================
# 6. Analytic PMP benchmark
# =============================================================================
def analytic_pmp_solution(cfg: Dict[str, Any]) -> Tuple[np.ndarray, np.ndarray]:
    dt = get_dt(cfg)
    N = int(cfg["N_steps"])
    H = int(cfg["H_delay"])

    w = build_a1_kernel_np(cfg)

    A = np.zeros((H, H), dtype=np.float64)
    A[0, 0] = 1.0 + dt * float(cfg["a0"]) + dt * float(w[0])
    A[0, 1:] = dt * w[1:]

    for i in range(1, H):
        A[i, i - 1] = 1.0

    Y = np.zeros((N + 1, H), dtype=np.float64)
    Y[N, 0] = float(cfg["S0"])

    AT = A.T
    for n in reversed(range(N)):
        Y[n] = AT @ Y[n + 1]

    v_star = np.zeros(N + 1, dtype=np.float64)
    for n in range(N + 1):
        v_un = -(float(cfg["b0"]) / (2.0 * float(cfg["R"]))) * Y[n, 0]
        v_star[n] = max(0.0, v_un)

    t_grid = np.linspace(0.0, float(cfg["T"]), N + 1)
    return t_grid, v_star


# =============================================================================
# 7. Metrics / Plot
# =============================================================================
def compute_metrics(gt: np.ndarray, pred: np.ndarray) -> Tuple[float, float]:
    m = min(len(gt), len(pred))
    rmse = float(np.sqrt(np.mean((pred[:m] - gt[:m]) ** 2)))
    mae = float(np.mean(np.abs(pred[:m] - gt[:m])))
    return rmse, mae


def metric_mean_std(results: List[Dict[str, Any]], key: str) -> Tuple[float, float]:
    vals = np.array([r[key] for r in results], dtype=np.float64)
    mean = float(np.mean(vals))
    std = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
    return mean, std


def print_seedwise_table(results: List[Dict[str, Any]]) -> None:
    print("\n" + "#" * 90)
    print("Seed-wise Metrics")
    print("#" * 90)

    header = f"{'seed':>5} | {'Algorithm':<10} | {'RMSE':>12} | {'MAE':>12}"
    print(header)
    print("-" * len(header))

    algos = [
        ("LSTM-DPO", "lstm"),
        ("PGDPO", "pgdpo"),
    ]

    for r in results:
        for name, key in algos:
            print(
                f"{r['seed']:5d} | {name:<10} | "
                f"{r[f'{key}_rmse']:12.6e} | {r[f'{key}_mae']:12.6e}"
            )


def print_multiseed_summary(results: List[Dict[str, Any]]) -> None:
    print("\n" + "#" * 90)
    print("Multi-seed Summary: mean ± std over seeds")
    print("#" * 90)

    header = (
        f"{'Algorithm':<10} | "
        f"{'RMSE mean':>12} | {'RMSE std':>12} | "
        f"{'MAE mean':>12} | {'MAE std':>12}"
    )
    print(header)
    print("-" * len(header))

    algos = [
        ("LSTM-DPO", "lstm"),
        ("PGDPO", "pgdpo"),
    ]

    for name, key in algos:
        rmse_mean, rmse_std = metric_mean_std(results, f"{key}_rmse")
        mae_mean, mae_std = metric_mean_std(results, f"{key}_mae")

        print(
            f"{name:<10} | "
            f"{rmse_mean:12.6e} | {rmse_std:12.6e} | "
            f"{mae_mean:12.6e} | {mae_std:12.6e}"
        )


def print_projection_timing_summary(results: List[Dict[str, Any]]) -> None:
    print("\n" + "#" * 90)
    print("Projection Timing Summary: mean ± std over seeds")
    print("#" * 90)

    rows = [
        ("Total projection sec", "pgdpo_projection_total_sec"),
        ("Avg sec / step", "pgdpo_projection_avg_step_sec"),
        ("Min sec / step", "pgdpo_projection_min_step_sec"),
        ("Max sec / step", "pgdpo_projection_max_step_sec"),
    ]

    header = f"{'Metric':<28} | {'mean':>12} | {'std':>12}"
    print(header)
    print("-" * len(header))

    for name, key in rows:
        vals = np.array([r[key] for r in results], dtype=np.float64)
        mean = float(np.mean(vals))
        std = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0

        print(f"{name:<28} | {mean:12.6e} | {std:12.6e}")


def curve_mean_std(curves: List[np.ndarray]) -> Tuple[np.ndarray, np.ndarray]:
    arr = np.stack(curves, axis=0).astype(np.float64)
    mean = np.mean(arr, axis=0)
    std = np.std(arr, axis=0, ddof=1) if arr.shape[0] > 1 else np.zeros_like(mean)
    return mean, std


def plot_controls_band(
    t_grid: np.ndarray,
    v_star: np.ndarray,
    curves_by_algo: Dict[str, List[np.ndarray]],
    save_path: str,
) -> None:
    fig, ax = plt.subplots(figsize=(7, 5))

    ax.plot(t_grid, v_star, "k-", lw=2.2, label="Benchmark")

    style = {
        "LSTM-DPO": {
            "color": "tab:blue",
            "mean_ls": ":",
            "label": "LSTM-DPO",
        },
        "PGDPO": {
            "color": "tab:red",
            "mean_ls": "--",
            "label": "PGDPO",
        },
    }

    for algo in ["LSTM-DPO", "PGDPO"]:
        mean_curve, std_curve = curve_mean_std(curves_by_algo[algo])
        upper = mean_curve + std_curve
        lower = mean_curve - std_curve

        color = style[algo]["color"]

        ax.plot(
            t_grid,
            mean_curve,
            linestyle=style[algo]["mean_ls"],
            color=color,
            lw=2.0,
            label=style[algo]["label"],
        )

        ax.fill_between(
            t_grid,
            lower,
            upper,
            color=color,
            alpha=0.16,
            linewidth=0.0,
        )

        # band upper/lower boundaries as dotted curves
        ax.plot(t_grid, upper, linestyle=":", color=color, lw=1.25, alpha=0.95)
        ax.plot(t_grid, lower, linestyle=":", color=color, lw=1.25, alpha=0.95)

    ax.set_xlabel("Time")
    ax.set_ylabel("Advertising Expenditure")
    ax.legend()
    ax.grid(alpha=0.1)
    fig.tight_layout()

    #fig.savefig(save_path, format="pdf", bbox_inches="tight")
    #print(f"[Saved] {save_path}")

    plt.show()
    plt.close(fig)


# =============================================================================
# 8. Single seed comparison
# =============================================================================
def run_single_seed(seed: int, cfg_base: Dict[str, Any]) -> Dict[str, Any]:
    cfg = cfg_for_seed(cfg_base, seed)
    set_seed(seed)

    print("\n" + "=" * 100)
    print(
        f"[RUN] seed={seed} | T={cfg['T']}, N={cfg['N_steps']}, "
        f"dt={get_dt(cfg):.4f}, H={cfg['H_delay']}, delta={cfg['delta']}, device={DEVICE}"
    )
    print("=" * 100)

    t0 = time.time()

    policy_pg = LSTMPolicy(input_size=2, hidden_size=int(cfg["hidden"]))
    policy_pg = warmup_train_pg(policy_pg, cfg)

    # analytic_pmp_solution returns N+1 points: t_0, ..., t_N.
    # For control-trajectory evaluation, exclude terminal t_N = T.
    t_grid_full, v_star_full = analytic_pmp_solution(cfg)

    N = int(cfg["N_steps"])
    assert len(t_grid_full) == N + 1
    assert len(v_star_full) == N + 1

    t_decision = t_grid_full[:-1]
    v_star_decision = v_star_full[:-1]

    policy_pg.eval()
    rollout = rollout_pg_deterministic(policy_pg, cfg)

    proj_out = compute_pgdpo_projection_with_timing(
        policy_pg=policy_pg,
        cfg=cfg,
        rollout=rollout,
    )

    v_lstm_np = proj_out["v_lstm"]
    v_pgdpo_np = proj_out["v_pgdpo"]
    proj_timing = proj_out["timing"]

    m = min(len(v_star_decision), len(v_lstm_np), len(v_pgdpo_np), len(t_decision))

    t_m = t_decision[:m]
    v_star_m = v_star_decision[:m]
    v_lstm_m = v_lstm_np[:m]
    v_pgdpo_m = v_pgdpo_np[:m]

    lstm_rmse, lstm_mae = compute_metrics(v_star_m, v_lstm_m)
    pgdpo_rmse, pgdpo_mae = compute_metrics(v_star_m, v_pgdpo_m)

    elapsed_sec = float(time.time() - t0)

    print("\n[Grid Check: terminal t=T excluded]")
    print(f"Full benchmark length       : {len(t_grid_full)}")
    print(f"Decision-grid length        : {len(t_m)}")
    print(f"LSTM-DPO length             : {len(v_lstm_m)}")
    print(f"PGDPO length                : {len(v_pgdpo_m)}")
    print(f"Last plotted/evaluated time : {t_m[-1]:.6f}")
    print(f"Excluded terminal time      : {t_grid_full[-1]:.6f}")
    print(f"Benchmark v*(last decision) : {v_star_full[-2]:.6f}")
    print(f"Benchmark v*(terminal)      : {v_star_full[-1]:.6f}")

    print("\n[Seed Metrics: decision grid only]")
    print(f"RMSE & MAE (LSTM-DPO): {lstm_rmse:.6f}, {lstm_mae:.6f}")
    print(f"RMSE & MAE (PGDPO)   : {pgdpo_rmse:.6f}, {pgdpo_mae:.6f}")

    print("\n[Projection Timing]")
    print(f"PGDPO projection steps        : {proj_timing['pgdpo_projection_n_steps']}")
    print(f"PGDPO projection total time   : {proj_timing['pgdpo_projection_total_sec']:.6f} sec")
    print(f"PGDPO projection avg/step     : {proj_timing['pgdpo_projection_avg_step_sec']:.6f} sec")
    print(f"PGDPO projection min/step     : {proj_timing['pgdpo_projection_min_step_sec']:.6f} sec")
    print(f"PGDPO projection max/step     : {proj_timing['pgdpo_projection_max_step_sec']:.6f} sec")

    print(f"[Elapsed] {elapsed_sec:.2f} sec")

    result = {
        "seed": int(seed),
        "elapsed_sec": elapsed_sec,

        "t_grid": t_m,
        "v_star": v_star_m,
        "v_lstm": v_lstm_m,
        "v_pgdpo": v_pgdpo_m,

        "lstm_rmse": lstm_rmse,
        "lstm_mae": lstm_mae,
        "pgdpo_rmse": pgdpo_rmse,
        "pgdpo_mae": pgdpo_mae,

        "pgdpo_projection_total_sec": proj_timing["pgdpo_projection_total_sec"],
        "pgdpo_projection_avg_step_sec": proj_timing["pgdpo_projection_avg_step_sec"],
        "pgdpo_projection_min_step_sec": proj_timing["pgdpo_projection_min_step_sec"],
        "pgdpo_projection_max_step_sec": proj_timing["pgdpo_projection_max_step_sec"],
        "pgdpo_projection_step_times": proj_timing["pgdpo_projection_step_times"],
        "pgdpo_projection_n_steps": proj_timing["pgdpo_projection_n_steps"],
    }

    del policy_pg
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


# =============================================================================
# 9. Multi-seed driver
# =============================================================================
def run_multiseed_experiment(cfg: Dict[str, Any]) -> Dict[str, Any]:
    os.makedirs(cfg["outdir"], exist_ok=True)

    results = []

    for seed in cfg["seeds"]:
        results.append(run_single_seed(seed, cfg))

    print_seedwise_table(results)
    print_multiseed_summary(results)
    print_projection_timing_summary(results)

    ref = results[0]

    curves_by_algo = {
        "LSTM-DPO": [r["v_lstm"] for r in results],
        "PGDPO": [r["v_pgdpo"] for r in results],
    }

    fig_path = os.path.join(
        cfg["outdir"],
        f"benchmark3_control_multiseed_band_seeds{cfg['seeds'][0]}to{cfg['seeds'][-1]}.pdf",
    )

    plot_controls_band(
        t_grid=ref["t_grid"],
        v_star=ref["v_star"],
        curves_by_algo=curves_by_algo,
        save_path=fig_path,
    )

    zip_path = os.path.join(
        cfg["outdir"],
        "benchmark3_multiseed_figures.zip",
    )
    '''
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        for fname in os.listdir(cfg["outdir"]):
            if fname.endswith(".pdf"):
                full_path = os.path.join(cfg["outdir"], fname)
                zipf.write(full_path, arcname=fname)

    print(f"[ZIP saved] {zip_path}")
    '''
    return {
        "cfg": cfg,
        "results": results,
        "fig_path": fig_path,
        #"zip_path": zip_path,
    }


# =============================================================================
# 10. Execute
# =============================================================================
RESULT = run_multiseed_experiment(CFG)
